In [2]:
!pip install torch torchvision matplotlib


In [3]:
import torch
from torchvision import datasets, transforms, models
from torch import nn, optim
from torch.utils.data import DataLoader


In [28]:
!unzip datasets.zip


Archive:  datasets.zip
   creating: datasets/
   creating: datasets/train/
   creating: datasets/train/adhirasam/
  inflating: datasets/train/adhirasam/02d09e872d.jpg  
  inflating: datasets/train/adhirasam/02f2e49039.jpg  
  inflating: datasets/train/adhirasam/06c639bab2.jpg  
  inflating: datasets/train/adhirasam/08b8f8efc1.jpg  
  inflating: datasets/train/adhirasam/0cd70ffc10.jpg  
  inflating: datasets/train/adhirasam/0d09b0e580.jpg  
  inflating: datasets/train/adhirasam/10f608ecc4.jpg  
  inflating: datasets/train/adhirasam/12b1eb8026.jpg  
  inflating: datasets/train/adhirasam/13a2cfd13f.jpg  
  inflating: datasets/train/adhirasam/13d63deed2.jpg  
  inflating: datasets/train/adhirasam/14a2340ece.jpg  
  inflating: datasets/train/adhirasam/14d893495c.jpg  
  inflating: datasets/train/adhirasam/1a77e9cec2.jpg  
  inflating: datasets/train/adhirasam/1a9346fd1e.jpg  
  inflating: datasets/train/adhirasam/1fe5414330.jpg  
  inflating: datasets/train/adhirasam/2f6cc87911.jpg  
  infl

In [32]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor()
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])



In [33]:
train_data = datasets.ImageFolder("datasets/train", transform=train_transform)
val_data = datasets.ImageFolder("datasets/val", transform=val_transform)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32, shuffle=False)

num_classes = len(train_data.classes)
print("Classes:", num_classes)


Classes: 80


In [34]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.mobilenet_v2(pretrained=True)
model.classifier[1] = nn.Linear(1280, num_classes)
model = model.to(device)


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 114MB/s]


In [35]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)


In [36]:
epochs = 10

for epoch in range(epochs):
    model.train()
    train_loss = 0

    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    model.eval()
    correct, total = 0, 0

    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    acc = 100 * correct / total
    print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.2f} | Val Acc: {acc:.2f}%")


Epoch 1/10 | Train Loss: 380.43 | Val Acc: 41.88%
Epoch 2/10 | Train Loss: 240.71 | Val Acc: 53.00%
Epoch 3/10 | Train Loss: 166.34 | Val Acc: 61.00%
Epoch 4/10 | Train Loss: 123.15 | Val Acc: 66.38%
Epoch 5/10 | Train Loss: 92.86 | Val Acc: 66.00%
Epoch 6/10 | Train Loss: 70.39 | Val Acc: 66.88%
Epoch 7/10 | Train Loss: 53.04 | Val Acc: 68.38%
Epoch 8/10 | Train Loss: 39.32 | Val Acc: 69.25%
Epoch 9/10 | Train Loss: 28.51 | Val Acc: 69.00%
Epoch 10/10 | Train Loss: 20.99 | Val Acc: 68.25%


In [37]:
torch.save(model.state_dict(), "food_classifier.pth")


In [38]:
with open("classes.txt", "w") as f:
    for c in train_data.classes:
        f.write(c + "\n")
